In [6]:
import sympy as sp
from SymEigen import *
import diff_friction
import sys
sys.path.append('../')
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction('__host__ __device__')
Gen.DisableLatexComment()

N = 3
Xn = Eigen.Matrix('Xn', 3 * N, 1)
X = Eigen.Matrix('X', 3 * N, 1)
dX = X - Xn
dfdtau = Eigen.Matrix('dfdtau', 2, 1)

Cl = Gen.Closure(Xn)

P = Xn[0:3, :]
E0 = Xn[3:6, :]
E1 = Xn[6:9, :]

e = E1 - E0
r = P - E0
beta = e.dot(r) / e.dot(e)
beta

((Xn(0) - Xn(3))*(-Xn(3) + Xn(6)) + (Xn(1) - Xn(4))*(-Xn(4) + Xn(7)) + (Xn(2) - Xn(5))*(-Xn(5) + Xn(8)))/((-Xn(3) + Xn(6))**2 + (-Xn(4) + Xn(7))**2 + (-Xn(5) + Xn(8))**2)

In [7]:
e1 = Eigen.Normalize(e)
e2 =Eigen.Normalize(e1.cross(r))

beta_0 = 1 - beta
beta_1 = beta

In [8]:
T = sp.Matrix.zeros(2, 3 * N)

T[0, 0:3] = e1.T
T[1, 0:3] = e2.T

T[0, 3:6] = -beta_0 * e1.T
T[1, 3:6] = -beta_0 * e2.T

T[0, 6:9] = -beta_1 * e1.T
T[1, 6:9] = -beta_1 * e2.T

In [9]:
Cl = Gen.Closure(Xn)
content = ''
content += Cl('TR', T) + '\n'

In [10]:
dTdXn = diff_friction.compute_dTdXn(T, Xn)
dTdXn.shape

(9, 2, 9)

In [11]:
A, B = diff_friction.compute_AB(dTdXn, dX, dfdtau)
A.shape, B.shape

((2, 9), (9, 9))

In [12]:
Cl = Gen.Closure(Xn, X)
content += Cl('An', A) + '\n'

In [13]:
Cl = Gen.Closure(Xn, X, dfdtau)
content += Cl('Bn', B) + '\n'

In [14]:
filename = 'ipc_diff_friction_PE'
cuda_backend_source_dir = backend_source_dir('cuda')
f = open(f'{cuda_backend_source_dir}/contact_system/contact_models/sym/{filename}.inl', 'w')
f.write(content)
f.close()